# Bronze to Silver

This notebook follows one direct flow:

**Bronze files → `parse_aemo()` → four structured views → SQL → two Silver tables → validation**

Monthly, Daily and Current are merged into one actual-demand table. AEMO forecasts remain separate because each target time appears in several forecast runs.

In [0]:
from pyspark.sql import functions as F

## Parse the AEMO files

AEMO CSVs contain several record groups, not one ordinary header and table. `spark.read.text()` preserves each line so the parser can find the exact `I` row that defines a group and retain only its matching `D` rows.

The row pattern must be explicit: `DISPATCH,REGIONSUM` contains actual demand, while `P5MIN,REGIONSOLUTION` contains AEMO forecasts.

In [0]:
def parse_aemo(
    path,
    row_pattern,
    time_column,
    forecast_time_column=None,
    filter_runno=False,
):
    raw = spark.read.text(path)

    header_row = (
        raw
        .filter(F.col("value").startswith(f"I,{row_pattern},"))
        .select(F.split("value", ",").alias("fields"))
        .first()
    )

    if header_row is None:
        raise ValueError(f"No I row found for {row_pattern} in {path}")

    header = header_row["fields"]
    time_i = header.index(time_column)
    region_i = header.index("REGIONID")
    intervention_i = header.index("INTERVENTION")
    demand_i = header.index("TOTALDEMAND")

    rows = (
        raw
        .filter(F.col("value").startswith(f"D,{row_pattern},"))
        .withColumn("fields", F.split("value", ","))
        .filter(F.col("fields")[region_i] == "VIC1")
        .filter(F.col("fields")[intervention_i].cast("int") == 0)
    )

    if filter_runno:
        runno_i = header.index("RUNNO")
        rows = rows.filter(F.col("fields")[runno_i].cast("int") == 1)

    time = F.to_timestamp(
        F.regexp_replace(F.col("fields")[time_i], '"', ""),
        "yyyy/MM/dd HH:mm:ss",
    ).alias("time")

    demand = (
        F.regexp_replace(F.col("fields")[demand_i], '"', "")
        .cast("double")
        .alias("demand")
    )

    if forecast_time_column is None:
        return rows.select(time, demand)

    forecast_time_i = header.index(forecast_time_column)
    forecast_time = F.to_timestamp(
        F.regexp_replace(F.col("fields")[forecast_time_i], '"', ""),
        "yyyy/MM/dd HH:mm:ss",
    ).alias("forecast_time")

    return rows.select(forecast_time, time, demand)

## Actual demand

Monthly, Daily and Current contain the same observed VIC1 demand delivered through different AEMO layers. Each call returns `time | demand` and applies `RUNNO = 1`.

In [0]:
bronze_path = "/Volumes/workspace/default/aemo_mlops_volume/bronze"

monthly = parse_aemo(
    f"{bronze_path}/monthly_uncompressed/*.CSV",
    row_pattern="DISPATCH,REGIONSUM",
    time_column="SETTLEMENTDATE",
    filter_runno=True,
)

daily = parse_aemo(
    f"{bronze_path}/daily_uncompressed/*.CSV",
    row_pattern="DISPATCH,REGIONSUM",
    time_column="SETTLEMENTDATE",
    filter_runno=True,
)

current = parse_aemo(
    f"{bronze_path}/current_uncompressed/*.CSV",
    row_pattern="DISPATCH,REGIONSUM",
    time_column="SETTLEMENTDATE",
    filter_runno=True,
)

## AEMO demand forecast

P5MIN produces a new forecast run every five minutes. `forecast_time` is when AEMO produced the run; `time` is the future interval being predicted. The pair `(forecast_time, time)` identifies one forecast observation.

In [0]:
forecast = parse_aemo(
    [
        f"{bronze_path}/forecast/archive_uncompressed/*.CSV",
        f"{bronze_path}/forecast/current_uncompressed/*.CSV",
    ],
    row_pattern="P5MIN,REGIONSOLUTION",
    time_column="INTERVAL_DATETIME",
    forecast_time_column="RUN_DATETIME",
)

## Create the structured views

After these views are created, the irregular AEMO file structure is no longer relevant. The remaining work is ordinary SQL.

In [0]:
monthly.createOrReplaceTempView("monthly")
daily.createOrReplaceTempView("daily")
current.createOrReplaceTempView("current")
forecast.createOrReplaceTempView("forecast")

## Build the actual Silver table

Monthly, Daily and Current describe the same demand timeline and are merged into `demand_vic_5min`. When a time appears more than once, the project assumption prefers Current, then Daily, then Monthly.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.demand_vic_5min
USING DELTA
AS

WITH all_actual AS (
    SELECT time, demand, 1 AS priority FROM monthly
    UNION ALL
    SELECT time, demand, 2 AS priority FROM daily
    UNION ALL
    SELECT time, demand, 3 AS priority FROM current
),

ranked AS (
    SELECT
        time,
        demand,
        ROW_NUMBER() OVER (
            PARTITION BY time
            ORDER BY priority DESC
        ) AS row_number
    FROM all_actual
)

SELECT time, demand
FROM ranked
WHERE row_number = 1

## Build the forecast Silver table

Archive and Current forecast files can overlap. `DISTINCT` removes only identical rows and preserves the full forecast trajectory.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.aemo_demand_forecast_vic_5min
USING DELTA
AS

SELECT DISTINCT
    forecast_time,
    time,
    demand
FROM forecast

## Validate conflicting values

Exact duplicates are harmless. The first query checks the overlapping actual sources before priority is applied; the second checks forecast keys after exact deduplication. Empty results mean no conflicts were found.

In [0]:
%sql
WITH all_actual AS (
    SELECT time, demand FROM monthly
    UNION ALL
    SELECT time, demand FROM daily
    UNION ALL
    SELECT time, demand FROM current
)

SELECT
    time,
    SORT_ARRAY(COLLECT_SET(demand)) AS conflicting_values
FROM all_actual
GROUP BY time
HAVING COUNT(DISTINCT demand) > 1
ORDER BY time

In [0]:
%sql
SELECT
    forecast_time,
    time,
    SORT_ARRAY(COLLECT_SET(demand)) AS conflicting_values
FROM workspace.default.aemo_demand_forecast_vic_5min
GROUP BY forecast_time, time
HAVING COUNT(DISTINCT demand) > 1
ORDER BY forecast_time, time